In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  
from sklearn.decomposition import PCA
import os
import re
from pathlib import Path
from zipfile import ZipFile
import pandas as pd
import seaborn as sns
from scipy.optimize import curve_fit
from sklearn.cluster import AgglomerativeClustering
import statistics
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster, leaves_list
import csv
import plotly.graph_objects as go

In [ ]:
from joblib import Parallel, delayed
from sklearn.utils import shuffle
import statistics
def intersection_branch_distance(dict1,dict2): #issa metric from deeds
    branch_dist = 0
    for n in list(set(dict1).intersection(set(dict2))):
        branch_dist = branch_dist + (dict1[n]-dict2[n])**2
        #print(n,(dict1[n]-dict2[n])**2, branch_dist)
    return (np.sqrt(branch_dist))
def inter_mean_calculator(strain1, strain2, IBD_matrix):
    return np.mean(IBD_matrix[np.ix_(strain1, strain2)])

def intra_mean_calculator(strain, IBD_matrix):
    return np.mean(IBD_matrix[np.ix_(strain, strain)])

def inter_intra_distance(emb_pair, IBD_matrix):
    #calculate mean intrastrain IBD using (n_a*IBD_a + n_b*IBD_b)/(n_a + n_b)
    '''if emb_pair[0] == emb_pair[1]:
        return 0'''
    intra1 = intra_mean_calculator(emb_pair[0], IBD_matrix)
    intra2 = intra_mean_calculator(emb_pair[1], IBD_matrix)
    intra_strain = (len(emb_pair[0])*intra1 + len(emb_pair[1])*intra2)/(len(emb_pair[0])+ len(emb_pair[1]))
    #average out the interstrain IBD overall for all embryos
    inter_strain = inter_mean_calculator(emb_pair[0], emb_pair[1], IBD_matrix)
    inter_intra = inter_strain - intra_strain
    return inter_intra

def shuffler_distance(strain1, strain2, IBD_matrix):
    #print(len(strain1))
    set = strain1 + strain2
    shuffled = shuffle(set)
    #length = len(set) // 2
    set1 = shuffled[:len(strain1)]
    #print(len(set1))
    set2 = shuffled[len(strain1):]
    #print(len(set2))
    distance = inter_intra_distance([set1, set2], IBD_matrix)
    return distance

#primary permutation code 
def permutation_worker(strain_map, index_map, IBD_matrix, ref_matrix, strains, n_iter):
    local_count = np.zeros((len(strains), len(strains)))
    for _ in range(n_iter):
        for n in strain_map:
            for m in strain_map:
                shuffled_val = shuffler_distance(strain_map[n], strain_map[m], IBD_matrix)
                if shuffled_val > ref_matrix[index_map[n], index_map[m]]:
                    local_count[index_map[n], index_map[m]] += 1
                    
    return local_count

def permutation_test_parallel(IBD_matrix, strain_map, index_map, strains, its, n_jobs=-1):
    ref_matrix = np.zeros((len(strains), len(strains)))
    
    for n in strain_map:
        for m in strain_map:
            ref_matrix[index_map[n], index_map[m]] = inter_intra_distance(
                [strain_map[n], strain_map[m]], IBD_matrix
            )

    #split iterations across cores to save time
    n_jobs = os.cpu_count() if n_jobs == -1 else n_jobs
    chunk = its // n_jobs
    chunks = [chunk] * n_jobs
    chunks[0] += its % n_jobs  #remainder

    results = Parallel(n_jobs=n_jobs)(
        delayed(permutation_worker)(
            strain_map, index_map, IBD_matrix, ref_matrix, strains, c
        )
        for c in chunks
    )

    #combine results
    count_matrix = np.sum(results, axis=0)

    p_matrix = np.zeros((len(strains), len(strains)))
    for i in range(len(strains)):
        for j in range(i, len(strains)):
            p_matrix[i, j] = (count_matrix[i, j] + 1) / (its + 1)
    #only computed upper triangle to save time, mirror for the return
    p_matrix = p_matrix + p_matrix.T - np.diag(np.diag(p_matrix))
    return p_matrix


In [ ]:
import math
def normalized_decomposition_bd(df):
    M = df.values
    n = M.shape[0]
    #output matrix initialization
    parallel = np.zeros((n,n))
    perp = np.zeros((n,n))
    #iterate through pairwise row comparisons
    for i in range(n):
        for j in range(n):
            #branch distance squared (hypotenuse)
            d2 = np.sum((M[i] - M[j])**2)
            #convert to unit vectors to calculate mean
            u = M[i] / np.linalg.norm(M[i])
            v = M[j] / np.linalg.norm(M[j])
            #mean vector
            m = (u + v) / 2
            norm_m = np.linalg.norm(m)
            if norm_m == 0:
                parallel[i,j] = 0
                perp[i,j] = 0
                continue
            m = m / norm_m
            par = abs(np.dot(M[i] - M[j], m))
            #stil use original vectors for projections
            if d2 == 0: #when comparing identical embs/vectors 
                parallel[i,j] = 0
                perp[i,j] = 0
                continue
            #variance from parallel(scale) and perpendicular(not scale)
            parallel[i,j] = par
            perp[i,j] = math.sqrt(d2 - par**2)
    #output dfs
    parallel_df = pd.DataFrame(parallel, index=df.index, columns=df.index)
    perp_df = pd.DataFrame(perp, index=df.index, columns=df.index)
    return parallel_df, perp_df

normalized_scale, normalized_nonscale = normalized_decomposition_bd(df)
normalized_scale.to_csv('new_scale.csv')
normalized_nonscale.to_csv('new_nonscale.csv')

In [ ]:
def p_matrix_csv(motmat, namelist, filename):
#new list to store intersections + embryo names
    #print(motmat)
    array_title = [] 
    rows = cols = len(namelist)+1
    for _ in range(rows):
        array_title.append([0] * cols)
    for i in range(1, cols):
        array_title[0][i] = namelist[i-1]  
        array_title[i][0] = namelist[i-1] 
    for i in range(1, rows):
        for j in range(1, cols):
            array_title[i][j] = motmat[i-1][j-1]
    
    with open(filename, mode='w', newline='') as file:
        writer = csv.writer(file)  
        for row in array_title:
            writer.writerow(row)
    return

In [ ]:
strains = ['N2', 'CB4856', 'JIM113', 'MY23', 'DL238', 'ECA36', 'JU2001', 'XZ1516']
index_map = {}
count = 0
its = 100000
for n in strains:
    index_map[n] = count
    count += 1

In [ ]:
#add strain map for later use so that what strain corresponds to what index is easily hashable
from collections import defaultdict
ranges = [
    (0, 9,  "N2"),
    (9, 17, "CB4856"),
    (17, 27,"JIM113"),
    (27, 37,"MY23"),
    (37, 47,"DL238"),
    (47, 57,"ECA36"),
    (57, 67,"JU2001"),
    (67, 77,"XZ1516"),
]

strain_map = {}
ylabels = []

for start, end, label in ranges:
    for i in range(start, end):
        strain_map[i] = label
        ylabels.append(label)

grouped_dict = defaultdict(list)
for key, value in strain_map.items():
    grouped_dict[value].append(key)
permutation_map = grouped_dict

In [ ]:
import pickle
with open("fulldictlist.pkl", "rb") as f:
  fulldictlist = pickle.load(f)


In [ ]:
print(index_map)
whole_IBD = np.zeros((len(fulldictlist), len(fulldictlist)))
for n in range(len(fulldictlist)):
    for m in range(len(fulldictlist)):
        whole_IBD[n][m] = intersection_branch_distance(fulldictlist[n], fulldictlist[m])
matrix = permutation_test_parallel(whole_IBD, permutation_map, index_map, strains, its)

In [ ]:
p_matrix_csv(matrix, strains, 'whole_embryo.csv')

In [ ]:
its = 100000
parallel_p_matrix = permutation_test_parallel(normalized_scale.values, permutation_map, index_map, strains, its)
perpendicular_p_matrix = permutation_test_parallel(normalized_nonscale.values, permutation_map, index_map, strains, its)
p_matrix_csv(parallel_p_matrix,strains, 'whole_embryo_parallel.csv')
p_matrix_csv(perpendicular_p_matrix,strains,'whole_embryo_perpendicular.csv')